# 공개 데이터 SFT 1단계 — NuminaMath 필터링

## 이 노트북이 하는 일
NuminaMath-1.5(896k 문제)에서 우리 대회에 맞는 것만 골라내고, RFT 데이터와 섞어 학습 세트를 만듭니다.

```
NuminaMath-1.5 (896k)
   → 필터링 (정수 답 / 이미지 없음 / 유효)
   → 평가 문항 중복 제거 (검증 300 + 리더보드 831)
   → 샘플링 20k
   → RFT 5,465개와 혼합
   → mixed_sft.jsonl
```

## 왜 NuminaMath인가
대회 train 문항 127개에 `cdn.mathpix.com` 이미지 링크가 있는데, **NuminaMath와 같은 OCR 형식**입니다.
`Problem N.` / `Task N.` / `A1`·`G1`·`NT1` 같은 문제 번호 접두사도 일치합니다.
→ **대회 데이터가 NuminaMath 계열에서 파생된 것으로 보입니다.** 분포가 정확히 맞는 유일한 후보예요.

- 라이선스: **Apache 2.0** (무료·동등 접근 → 규칙 5.2 충족)
- 최종 제출 시 사용 데이터셋 목록에 **`AI-MO/NuminaMath-1.5` 명시 필요** (규칙 5.2c)

## ⚠️ 중복 제거를 반드시 하는 이유
검증셋 300문제가 학습에 섞이면 정확도가 0.85처럼 보입니다. 하지만 그건 **외운 걸 다시 맞힌 것**이에요.
그 숫자를 믿고 8/31에 갔다가 실제로 0.74가 나오면 되돌릴 수 없습니다.
리더보드 831문제도 같습니다 — 넣으면 순위는 오르지만 **최종 test에는 도움이 안 됩니다.**

## 실행 순서
`[1]` `[2]` → ⛔Restart → `[1]` → `[3]`~`[8]`  (GPU 불필요, 약 15분)


---
## [1] 설정 ▶️

**이 노트북은 GPU가 필요 없습니다.** Accelerator를 None으로 두면 할당량을 안 씁니다.
단, **Internet은 On**이어야 NuminaMath를 받을 수 있습니다.

In [ ]:
N_NUMINA   = 20000     # NuminaMath에서 뽑을 문제 수
MAX_SOL_CH = 3000      # 풀이 길이 상한(글자). 너무 긴 건 학습 비용만 늘림
MIN_SOL_CH = 100       # 너무 짧은 건 풀이라 보기 어려움
VALID_N    = 300       # 추론 노트북과 동일
SEED       = 42        # 추론 노트북과 동일
OUT_PATH   = "/kaggle/working/mixed_sft.jsonl"
print("GPU 불필요 / Internet On 필요")

---
## [2] 설치 ⏭️ 세션 안 껐으면 건너뛰기
---
## ⛔ 설치 후 Run → Restart Session → [1]부터
---

In [ ]:
!pip install -q -U datasets 2>&1 | tail -2
import datasets; print("datasets", datasets.__version__)

---
## [3] 평가 문항 목록 만들기 ▶️

**먼저 빼야 할 문제들을 확보합니다.** 나중에 하면 실수합니다.

- 검증셋 300문제: `train.sample(300, random_state=42)` — 추론 노트북과 완전히 동일
- 리더보드 831문제: 전부

비교는 **정규화된 문장**으로 합니다. 공백·대소문자·LaTeX 공백을 지워서, 표기가 조금 달라도 같은 문제면 잡히게요.

In [ ]:
import glob, os, re, pandas as pd

def find_csv(must_have, must_not=()):
    for p in sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True)):
        b = os.path.basename(p).lower()
        if all(k in b for k in must_have) and not any(k in b for k in must_not):
            return p

TRAIN_PATH = find_csv(["train"], must_not=["filtered","ids","leaderboard","test"])
BAD_PATH   = find_csv(["filtered","ids"])
LB_PATH    = find_csv(["leaderboard","filtered"]) or find_csv(["leaderboard"])
assert TRAIN_PATH and LB_PATH and TRAIN_PATH != BAD_PATH

train = pd.read_csv(TRAIN_PATH)
train = train[~train["id"].isin(set(pd.read_csv(BAD_PATH)["id"]))].reset_index(drop=True)
assert len(train) == 16373, f"16373이어야 하는데 {len(train)}"
lb = pd.read_csv(LB_PATH)

def norm(s):
    s = str(s).lower()
    s = re.sub(r"\\[a-z]+", " ", s)      # LaTeX 명령 제거
    s = re.sub(r"[^a-z0-9]+", "", s)     # 영숫자만 남김
    return s

valid_q = train.sample(VALID_N, random_state=SEED)["question"]
BLOCK = set(norm(q) for q in valid_q) | set(norm(q) for q in lb["question"])
BLOCK.discard("")

print(f"검증셋 {len(valid_q)} + 리더보드 {len(lb)} = 차단 목록 {len(BLOCK)}개")
print("첫 검증 문항 id:", train.sample(VALID_N, random_state=SEED).iloc[0]["id"], "(train-004925 여야 함)")

---
## [4] NuminaMath 다운로드 ▶️ 약 3~5분

531MB입니다. `streaming=False`로 전체를 받아 pandas로 다룹니다.

In [ ]:
from datasets import load_dataset

ds = load_dataset("AI-MO/NuminaMath-1.5", split="train")
nm = ds.to_pandas()
print(nm.shape)
print(nm["question_type"].value_counts())
print()
print(nm["source"].value_counts())

---
## [5] 필터링 ▶️

### 조건별 이유

| 조건 | 이유 |
|---|---|
| `question_type == "math-word-problem"` | 증명·객관식 제외. 우리 대회는 숫자 답 |
| `answer`가 정수로 파싱 | 대회는 **정답이 항상 정수** |
| 이미지 링크 없음 | 그림이 본문이면 텍스트만으론 못 품. 주최 측도 127개를 전부 걸러냈음 |
| `problem_is_valid`, `solution_is_valid` == Yes | 데이터셋 자체 품질 표시 |
| 풀이 길이 100~3000자 | 너무 짧으면 풀이가 아니고, 너무 길면 학습 비용만 늘어남 |
| **차단 목록 제외** | 검증셋·리더보드 문항 |

In [ ]:
import re
from collections import Counter

def extract_boxed(text):
    """Return the raw content inside the LAST \\boxed{...}, brace-balanced."""
    idx = text.rfind('\\boxed')
    if idx == -1:
        return None
    i = idx + len('\\boxed')
    while i < len(text) and text[i] == ' ':
        i += 1
    if i >= len(text):
        return None
    if text[i] != '{':                       # bare form: \boxed 15
        m = re.match(r'-?[\d,]+', text[i:])
        return m.group(0) if m else None
    depth, start = 0, i + 1
    while i < len(text):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[start:i]
        i += 1
    return None

def to_int(s):
    """LaTeX/text -> python int, or None. Never uses float(), so huge ints survive."""
    if s is None:
        return None
    s = str(s).strip()
    s = s.replace('{,}', '').replace('{\\,}', '')          # LaTeX thousands separator
    s = re.sub(r'\\(?:text|mathrm|mbox|textbf|textrm)\s*\{([^{}]*)\}', r'\1', s)
    for junk in ['\\!', '\\,', '\\;', '\\:', '\\ ', '\\left', '\\right',
                 '\\$', '$', '%', '~', '^\\circ', '\\%']:
        s = s.replace(junk, '')
    s = s.replace(',', '').replace(' ', '').strip()
    s = re.sub(r'[a-zA-Z]+$', '', s)                       # trailing unit: 42cm -> 42
    while len(s) > 1 and s[0] == '(' and s[-1] == ')':     # (\frac{100}{4}) -> \frac{100}{4}
        s = s[1:-1].strip()
    s = s.rstrip('.')
    if not s:
        return None
    m = re.fullmatch(r'\\[dt]?frac\{([-+]?\d+)\}\{([-+]?\d+)\}', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)/([-+]?\d+)', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)(?:\\times|\\cdot)10\^\{?(\d+)\}?', s)
    if m:
        return int(m.group(1)) * 10 ** int(m.group(2))
    if re.fullmatch(r'[-+]?\d+', s):
        return int(s)
    m = re.fullmatch(r'([-+]?\d+)\.0*', s)
    if m:
        return int(m.group(1))
    return None

def last_int(text):
    for c in reversed(re.findall(r'-?\d[\d,]*', text)):
        v = to_int(c)
        if v is not None:
            return v
    return None

def parse_answer(text):
    """None means 'this sample produced no usable integer' -> dropped from voting."""
    raw = extract_boxed(text)
    if raw is not None:
        return to_int(raw)          # boxed present but unparseable -> None, do NOT guess
    m = re.findall(r'(?:answer|Answer|ANSWER)\s*(?:is|:|=)+\s*\$?(-?[\d,]+)', text)
    if m:
        v = to_int(m[-1])
        if v is not None:
            return v
    return last_int(text)

def majority_vote(values, fallback=0):
    vals = [v for v in values if v is not None]
    if not vals:
        return fallback
    return Counter(vals).most_common(1)[0][0]


n0 = len(nm)
f = nm[nm["question_type"] == "math-word-problem"].copy()
print(f"math-word-problem            : {len(f):>7,}  ({len(f)/n0:.1%})")

f = f[(f["problem_is_valid"] == "Yes") & (f["solution_is_valid"] == "Yes")]
print(f"+ 유효성 Yes                  : {len(f):>7,}")

img = f["problem"].str.contains(r"!\[\]|cdn\.mathpix|\.jpg|\.png", regex=True, na=False) | \
      f["solution"].str.contains(r"!\[\]|cdn\.mathpix|\.jpg|\.png", regex=True, na=False)
f = f[~img]
print(f"+ 이미지 없음                  : {len(f):>7,}")

f["ans_int"] = f["answer"].map(to_int)
f = f[f["ans_int"].notna()]
print(f"+ 정수 답                     : {len(f):>7,}")

sl = f["solution"].str.len()
f = f[(sl >= MIN_SOL_CH) & (sl <= MAX_SOL_CH)]
print(f"+ 풀이 길이 {MIN_SOL_CH}~{MAX_SOL_CH}자        : {len(f):>7,}")

f["nq"] = f["problem"].map(norm)
hit = f["nq"].isin(BLOCK)
print(f"\n⚠️ 평가 문항과 중복: {hit.sum()}개 발견 → 제거")
f = f[~hit]
print(f"+ 중복 제거 후                 : {len(f):>7,}")

f = f.drop_duplicates(subset="nq")
print(f"+ 자체 중복 제거 후            : {len(f):>7,}")

---
## [6] 난이도 균형 잡아 샘플링 ▶️

`source`별로 난이도가 다릅니다.

| source | 성격 |
|---|---|
| `orca_math`, `cn_k12`, `synthetic_math` | 쉬운 문장제 (GSM8K급) |
| `olympiads`, `amc_aime`, `aops_forum` | 올림피아드·경시 |

대회 데이터도 이 둘이 섞여 있습니다(초등 문장제 ~ AIME). **한쪽으로 쏠리지 않게** source별 비율을 유지해서 뽑습니다.

In [ ]:
import numpy as np

print("필터 통과분의 source 분포")
print(f["source"].value_counts())

# source별 비율을 유지하며 N_NUMINA개 추출
frac = min(1.0, N_NUMINA / len(f))
samp = (f.groupby("source", group_keys=False)
          .apply(lambda g: g.sample(max(1, int(round(len(g) * frac))), random_state=SEED)))
samp = samp.sample(min(N_NUMINA, len(samp)), random_state=SEED).reset_index(drop=True)

print(f"\n추출: {len(samp):,}개")
print(samp["source"].value_counts())

---
## [7] 형식 정규화 + RFT와 혼합 ▶️

### 왜 정규화가 필요한가
NuminaMath 풀이는 OCR된 교과서 문체라 **`\boxed{}`로 끝나지 않는 경우가 많습니다.**
그대로 학습하면 모델이 `\boxed{}` 습관을 잃고 파싱 실패율이 치솟습니다.

→ 풀이 끝에 `\boxed{정답}`이 없으면 **한 줄 덧붙입니다.**

### 왜 RFT를 섞는가
NuminaMath는 문체가 모델과 다릅니다. RFT 데이터는 **모델 자신이 쓴 풀이**라 분포가 정확히 맞아요.
A/B 실험에서 RFT가 정확도는 못 올렸지만 **파싱 실패율을 1.70% → 0.99%로 낮췄습니다.**
형식 안정성 담당으로 남겨둡니다.

In [ ]:
import json, random

records = []

# NuminaMath
n_fixed = 0
for _, r in samp.iterrows():
    sol, ans = str(r["solution"]).strip(), int(r["ans_int"])
    if parse_answer(sol) != ans:                       # boxed가 없거나 값이 다르면
        sol = sol.rstrip() + f"\n\nThe final answer is $\\boxed{{{ans}}}$."
        n_fixed += 1
    records.append({"question": str(r["problem"]).strip(), "solution": sol,
                    "answer": ans, "src": "numina"})
print(f"NuminaMath {len(records):,}개 (boxed 보정 {n_fixed:,}개, {n_fixed/len(records):.1%})")

# RFT 혼합
rft_path = next((p for p in glob.glob("/kaggle/input/**/*.jsonl", recursive=True)
                 if "rft" in os.path.basename(p).lower()), None)
if rft_path:
    rft = [json.loads(l) for l in open(rft_path, encoding="utf-8")]
    easy = [r for r in rft if r["n_correct"] == 4]
    hard = [r for r in rft if r["n_correct"] < 4]
    random.Random(SEED).shuffle(easy)
    use = hard + easy[:1800]
    for r in use:
        records.append({"question": r["question"], "solution": r["solution"],
                        "answer": int(r["answer"]), "src": "rft"})
    print(f"RFT {len(use):,}개 추가")
else:
    print("[경고] RFT jsonl 을 못 찾음 — NuminaMath만 사용")

random.Random(SEED).shuffle(records)
print(f"\n최종 학습 세트: {len(records):,}개")
from collections import Counter
print(Counter(r["src"] for r in records))

---
## [8] 저장 + 확인 ▶️

저장 후 **반드시 다운로드**하고 Kaggle Dataset으로 올리세요.
`/kaggle/working/`은 세션 종료 시 사라집니다.

In [ ]:
with open(OUT_PATH, "w", encoding="utf-8") as fp:
    for r in records:
        fp.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"저장: {OUT_PATH} ({os.path.getsize(OUT_PATH)/1e6:.1f} MB, {len(records):,}줄)")

print("\n" + "="*70)
print("NuminaMath 샘플")
print("="*70)
ex = next(r for r in records if r["src"] == "numina")
print("[문제]", ex["question"][:250])
print("\n[풀이]", ex["solution"][-400:])
print("\n[정답]", ex["answer"])

print("\n" + "="*70)
print("최종 제출 시 명시할 외부 데이터셋 (규칙 5.2c)")
print("="*70)
print("  AI-MO/NuminaMath-1.5  (Apache 2.0, https://huggingface.co/datasets/AI-MO/NuminaMath-1.5)")

from IPython.display import FileLink
FileLink("mixed_sft.jsonl")